# Task10-JA: 日本語BERT (tohoku-nlp/bert-base-japanese-v3) 比較実験

英語版Task10のPhase 8 DistilBERT比較（`outputs/runs/phase8-bert-seed42/`）と同じ設計思想で、日本語版Full 800件・共通5-foldに対して日本語BERTをファインチューニングし、Core（TF-IDF+線形分類器）と比較します。

**このNotebookは、GPUのないサンドボックス環境でこのプロジェクトを自律実行した際に、ローカルCPUで実際に完走させたフルランのフォールバック/検証用として用意したものです。** ローカルCPU実行（`outputs/runs/phaseJA7-bert-seed42/manifest.json`の`execution_environment: "local_cpu_isolated_venv_python3.12"`）が既に完走している場合、このNotebookはGPU環境での再実行・再検証・より高速な再現に使えます。

## 必須の再現性チェック

実行前に、以下のセルで読み込んだデータ・Foldのhashが、Task10-JAプロジェクトが記録した値と一致することを必ず確認してください。一致しない場合は結果を提出物として使用しないでください。

- 期待されるFull data hash: `6d010d81e7d0dfc502eefb539a3523e70a0fb7f4c7fae909c9bdc338ca9fbf63`
- 期待されるFold artifact hash: `41a10ce176bcf1f0c545a4b41a144e2e449c11dc0f3d7756460af680076c3ffb`

## 使い方

1. Google ColabでGPUランタイム（ランタイム > ランタイムのタイプを変更 > GPU）を選択してください。
2. `data/raw/full_emails_ja.jsonl`と`outputs/folds/common_folds_ja.json`をアップロードしてください（このリポジトリのファイルです）。
3. 上から順にセルを実行してください。

In [ ]:
!pip install -q transformers torch scikit-learn pandas tqdm fugashi unidic-lite sentencepiece

In [ ]:
import hashlib
import json
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

# Colabにアップロードしたパスに合わせて調整してください。
data_path = Path("full_emails_ja.jsonl")
fold_path = Path("common_folds_ja.json")

EXPECTED_DATA_HASH = "6d010d81e7d0dfc502eefb539a3523e70a0fb7f4c7fae909c9bdc338ca9fbf63"
EXPECTED_FOLD_HASH = "41a10ce176bcf1f0c545a4b41a144e2e449c11dc0f3d7756460af680076c3ffb"


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


actual_data_hash = sha256_file(data_path)
actual_fold_hash = sha256_file(fold_path)
print("data hash match:", actual_data_hash == EXPECTED_DATA_HASH, actual_data_hash)
print("fold hash match:", actual_fold_hash == EXPECTED_FOLD_HASH, actual_fold_hash)
assert actual_data_hash == EXPECTED_DATA_HASH, "Full data does not match the approved hash -- stop."
assert actual_fold_hash == EXPECTED_FOLD_HASH, "Fold artifact does not match the approved hash -- stop."

## データとFoldの読み込み

In [ ]:
records = [json.loads(line) for line in data_path.read_text(encoding="utf-8").splitlines()]
records_by_id = {record["id"]: record for record in records}

fold_artifact = json.loads(fold_path.read_text(encoding="utf-8"))
n_splits = fold_artifact["metadata"]["n_splits"]
fold_rows = fold_artifact["records"]

LABELS = ["product_inquiry", "technical_issue", "billing", "account_support"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}

print(f"{len(records)} records, {n_splits} folds")

## Core比較セル: 同一`body_text`入力でのTF-IDF + LinearSVC

BERTとの比較で「モデル構造の差」と「入力処理（前処理）の差」を分離するため、Sudachi分かち書きを一切使わず、`body_text`を直接TF-IDFへ渡す条件（`BODY_RAW`）を用意します。これはCoreのJ0〜JC（Sudachi前処理あり）とは別の、BERTとの入力対等性のためだけの比較セルです。

In [ ]:
body_raw_rows = []
for fold_id in range(n_splits):
    this_fold = [row for row in fold_rows if row["fold_id"] == fold_id]
    train_ids = [row["sample_id"] for row in this_fold if row["split_role"] == "train"]
    val_ids = [row["sample_id"] for row in this_fold if row["split_role"] == "validation"]

    x_train = [records_by_id[i]["body_text"] for i in train_ids]
    y_train = [records_by_id[i]["label"] for i in train_ids]
    x_val = [records_by_id[i]["body_text"] for i in val_ids]
    y_val = [records_by_id[i]["label"] for i in val_ids]

    vectorizer = TfidfVectorizer()  # sklearn default tokenizer -- no Sudachi segmentation
    clf = LinearSVC(C=1.0)
    x_train_vec = vectorizer.fit_transform(x_train)
    clf.fit(x_train_vec, y_train)
    preds = clf.predict(vectorizer.transform(x_val))

    accuracy = accuracy_score(y_val, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val, preds, average="macro", labels=LABELS, zero_division=0
    )
    body_raw_rows.append(
        {"fold_id": fold_id, "accuracy": accuracy, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}
    )

body_raw_df = pd.DataFrame(body_raw_rows)
print("BODY_RAW (unsegmented body_text, sklearn default tokenizer) + LinearSVC:")
display(body_raw_df)
print("mean macro_f1:", body_raw_df["macro_f1"].mean())

## BERTファインチューニング: Dataset/学習/評価関数

In [ ]:
MODEL_NAME = "tohoku-nlp/bert-base-japanese-v3"
SEED = 42
MAX_LEN = 128
BATCH_SIZE = 16  # GPUメモリ不足の場合はここを下げ、GRAD_ACCUM_STEPSを増やしてください
GRAD_ACCUM_STEPS = 1
EPOCHS = 3
LEARNING_RATE = 2e-5

torch.manual_seed(SEED)


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt"
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item


def train_epoch(model, loader, optimizer, device, grad_accum_steps=1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / grad_accum_steps
        loss.backward()
        if (step + 1) % grad_accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        total_loss += outputs.loss.item()
    return total_loss / max(len(loader), 1)


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

## 5-fold ファインチューニングループ

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

fold_metrics = []
oof_rows = []
run_start = time.time()

for fold_id in range(n_splits):
    fold_start = time.time()
    this_fold = [row for row in fold_rows if row["fold_id"] == fold_id]
    train_ids = [row["sample_id"] for row in this_fold if row["split_role"] == "train"]
    val_ids = [row["sample_id"] for row in this_fold if row["split_role"] == "validation"]

    train_texts = [records_by_id[i]["body_text"] for i in train_ids]
    train_labels = [LABEL2ID[records_by_id[i]["label"]] for i in train_ids]
    val_texts = [records_by_id[i]["body_text"] for i in val_ids]
    val_labels = [LABEL2ID[records_by_id[i]["label"]] for i in val_ids]

    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS))
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    train_loader = DataLoader(TextDataset(train_texts, train_labels, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextDataset(val_texts, val_labels, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

    for epoch in range(EPOCHS):
        epoch_loss = train_epoch(model, train_loader, optimizer, device, GRAD_ACCUM_STEPS)
        print(f"fold={fold_id} epoch={epoch} loss={epoch_loss:.4f} elapsed={time.time()-fold_start:.1f}s")

    y_true_ids, y_pred_ids = evaluate(model, val_loader, device)
    y_true = [LABELS[i] for i in y_true_ids]
    y_pred = [LABELS[i] for i in y_pred_ids]

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)
    cw_precision, cw_recall, cw_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)

    row = {
        "fold_id": fold_id,
        "n_train": len(train_ids),
        "n_val": len(val_ids),
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "fold_seconds": time.time() - fold_start,
    }
    for label, p, r, f in zip(LABELS, cw_precision, cw_recall, cw_f1):
        row[f"precision_{label}"] = p
        row[f"recall_{label}"] = r
        row[f"f1_{label}"] = f
    fold_metrics.append(row)
    print("fold result:", row)

    for sample_id, true_label, predicted_label in zip(val_ids, y_true, y_pred):
        oof_rows.append(
            {
                "sample_id": sample_id,
                "condition": "bert_ja",
                "model": "tohoku-nlp-bert-base-japanese-v3",
                "fold_id": fold_id,
                "true_label": true_label,
                "predicted_label": predicted_label,
            }
        )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_seconds = time.time() - run_start
bert_df = pd.DataFrame(fold_metrics)
oof_df = pd.DataFrame(oof_rows)
print("total seconds:", total_seconds)
display(bert_df[["fold_id", "accuracy", "macro_precision", "macro_recall", "macro_f1"]])

## Core（best condition想定）とBERTの比較表

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "TF-IDF(body_text, no Sudachi) + LinearSVC",
            "accuracy": body_raw_df["accuracy"].mean(),
            "macro_precision": body_raw_df["macro_precision"].mean(),
            "macro_recall": body_raw_df["macro_recall"].mean(),
            "macro_f1": body_raw_df["macro_f1"].mean(),
        },
        {
            "model": "BERT (tohoku-nlp/bert-base-japanese-v3, fine-tuned)",
            "accuracy": bert_df["accuracy"].mean(),
            "macro_precision": bert_df["macro_precision"].mean(),
            "macro_recall": bert_df["macro_recall"].mean(),
            "macro_f1": bert_df["macro_f1"].mean(),
        },
    ]
)
print("=== same body_text input: naive TF-IDF vs BERT ===")
display(comparison)
print("Note: compare this BERT result against the Sudachi-segmented J0-JC Core conditions")
print("separately (outputs/runs/phaseJA4-core-seed42/metrics_summary.csv) -- the BODY_RAW row")
print("above isolates the input-processing effect, not Core's best achievable score.")

## 監査用成果物の書き出し

In [ ]:
import sklearn
import transformers as transformers_module

bert_df.to_csv("fold_metrics.csv", index=False)
oof_df.to_csv("predictions_oof.csv", index=False)
body_raw_df.to_csv("body_raw_core_comparison_metrics.csv", index=False)

manifest = {
    "run_id": "phaseJA7-bert-seed42-colab",
    "model_name": MODEL_NAME,
    "language": "ja",
    "data_hash": actual_data_hash,
    "fold_artifact_hash": actual_fold_hash,
    "n_splits": n_splits,
    "seed": SEED,
    "max_length": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "input_field": "body_text",
    "device": str(device),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers_module.__version__,
    "sklearn_version": sklearn.__version__,
    "total_training_seconds": total_seconds,
    "primary_metric": "macro_f1",
    "execution_environment": "google_colab",
}
with open("execution_manifest.json", "w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 成果物ファイルのダウンロード

In [ ]:
from google.colab import files

for file_path in ["fold_metrics.csv", "predictions_oof.csv", "body_raw_core_comparison_metrics.csv", "execution_manifest.json"]:
    files.download(file_path)